# SOLID Principles

**OOP-in-Python — Advanced**  
The five fundamental design principles for maintainable, scalable object-oriented software.

## What is SOLID?

| Letter | Principle | Core Idea |
|---|---|---|
| **S** | **S**ingle Responsibility | A class should have one, and only one, reason to change |
| **O** | **O**pen/Closed | Open for extension, closed for modification |
| **L** | **L**iskov Substitution | Subtypes must be substitutable for their base types |
| **I** | **I**nterface Segregation | Many specific interfaces are better than one general-purpose interface |
| **D** | **D**ependency Inversion | Depend on abstractions, not concretions |

---
## S — Single Responsibility Principle (SRP)

> *"A class should have one, and only one, reason to change."* — Robert C. Martin

Each class should do **one thing** and do it well.

In [ ]:
# VIOLATION: This class has multiple responsibilities
class Report:
    def __init__(self, data: list):
        self.data = data

    def generate(self) -> str:
        # Responsibility 1: Format data
        return "\n".join(f"- {item}" for item in self.data)

    def save_to_file(self, filename: str):
        # Responsibility 2: File I/O
        with open(filename, 'w') as f:
            f.write(self.generate())

    def send_email(self, recipient: str):
        # Responsibility 3: Email (and file + formatting!)
        print(f"Sending report to {recipient}")

In [ ]:
# CORRECT: Each class has a single responsibility
class ReportGenerator:
    """Responsibility: Format data into a report string."""
    def __init__(self, data: list):
        self.data = data

    def generate(self) -> str:
        return "\n".join(f"- {item}" for item in self.data)

class FileSaver:
    """Responsibility: Save content to files."""
    @staticmethod
    def save(content: str, filename: str):
        with open(filename, 'w') as f:
            f.write(content)

class EmailSender:
    """Responsibility: Send emails."""
    @staticmethod
    def send(subject: str, body: str, recipient: str):
        print(f"To: {recipient}\nSubject: {subject}\n\n{body}")

# Usage
report = ReportGenerator(["Item 1", "Item 2", "Item 3"])
content = report.generate()
FileSaver.save(content, "report.txt")
EmailSender.send("Monthly Report", content, "alice@example.com")
print("All responsibilities separated correctly.")

---
## O — Open/Closed Principle (OCP)

> *"Software entities should be open for extension, but closed for modification."*

Add new functionality by writing **new code**, not changing existing code.

In [ ]:
# VIOLATION: Adding a new shape requires modifying the AreaCalculator
class AreaCalculator:
    def calculate(self, shapes: list) -> float:
        total = 0.0
        for s in shapes:
            if s['type'] == 'circle':
                import math
                total += math.pi * s['radius'] ** 2
            elif s['type'] == 'rectangle':
                total += s['width'] * s['height']
            # Need to add another elif for each new shape!
        return total

In [ ]:
from abc import ABC, abstractmethod
import math

# CORRECT: Open for extension via new subclasses, closed for modification
class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        return math.pi * self.radius ** 2

class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

# New shape — NO modification to existing code!
class Triangle(Shape):
    def __init__(self, base: float, height: float):
        self.base = base
        self.height = height

    def area(self) -> float:
        return 0.5 * self.base * self.height

class AreaCalculator:
    @staticmethod
    def calculate(shapes: list[Shape]) -> float:
        return sum(s.area() for s in shapes)

shapes = [Circle(5), Rectangle(3, 4), Triangle(6, 4)]
print(f"Total area: {AreaCalculator.calculate(shapes):.2f}")

---
## L — Liskov Substitution Principle (LSP)

> *"If S is a subtype of T, then objects of type T may be replaced with objects of type S without altering the program's correctness."* — Barbara Liskov

Subclasses should **extend** base class behavior, not **break** it.

In [ ]:
# VIOLATION: Square breaks Rectangle's behavior
class Rectangle:
    def __init__(self):
        self.width = 0
        self.height = 0

    def set_width(self, w: int):
        self.width = w

    def set_height(self, h: int):
        self.height = h

    def area(self) -> int:
        return self.width * self.height

class Square(Rectangle):  # Violates LSP!
    def set_width(self, w: int):
        self.width = w
        self.height = w  # Side effect!

    def set_height(self, h: int):
        self.height = h
        self.width = h  # Side effect!

def test_rectangle(rect: Rectangle):
    rect.set_width(5)
    rect.set_height(10)
    # Client expects width=5, height=10, area=50
    assert rect.area() == 50, f"Area was {rect.area()}, expected 50"
    print(f"Area: {rect.area()} — OK")

test_rectangle(Rectangle())  # Passes
# test_rectangle(Square())  # Fails! Square violates LSP

In [ ]:
# CORRECT: Use composition or a common abstract base
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

class Square(Shape):
    def __init__(self, side: float):
        self.side = side

    def area(self) -> float:
        return self.side ** 2

# Now both Rectangle and Square are substitutable for Shape
shapes: list[Shape] = [Rectangle(5, 10), Square(5)]
for s in shapes:
    print(f"Area: {s.area()}")  # 50, 25

---
## I — Interface Segregation Principle (ISP)

> *"No client should be forced to depend on methods it does not use."*

Prefer **small, focused interfaces** over large, monolithic ones.

In [ ]:
# VIOLATION: Fat interface
class Worker(ABC):
    @abstractmethod
    def work(self): ...
    @abstractmethod
    def eat(self): ...    # Not all workers eat!
    @abstractmethod
    def sleep(self): ...   # Not all workers sleep!

class Human(Worker):
    def work(self): return "Working..."
    def eat(self): return "Eating..."
    def sleep(self): return "Sleeping..."

class Robot(Worker):
    def work(self): return "Working..."
    def eat(self): return "Robots don't eat!"  # Forced! Violates ISP
    def sleep(self): return "Robots don't sleep!"  # Forced!

In [ ]:
from abc import ABC, abstractmethod

# CORRECT: Segregated interfaces
class Workable(ABC):
    @abstractmethod
    def work(self): ...

class Eatable(ABC):
    @abstractmethod
    def eat(self): ...

class Sleepable(ABC):
    @abstractmethod
    def sleep(self): ...

class Human(Workable, Eatable, Sleepable):
    def work(self): return "Working..."
    def eat(self): return "Eating..."
    def sleep(self): return "Sleeping..."

class Robot(Workable):
    # Only implements what it actually needs!
    def work(self): return "Working 24/7..."

print(Human().work(), Human().eat())
print(Robot().work())

---
## D — Dependency Inversion Principle (DIP)

> *"High-level modules should not depend on low-level modules. Both should depend on abstractions."*

Depend on **interfaces/abstract classes**, not concrete implementations.

In [ ]:
# VIOLATION: High-level module depends directly on low-level module
class MySQLDatabase:
    def save(self, data: dict):
        print(f"Saving to MySQL: {data}")

class UserController:
    def __init__(self):
        self.db = MySQLDatabase()  # Tight coupling!

    def create_user(self, name: str):
        self.db.save({"name": name})

# Switching to MongoDB means modifying UserController!

In [ ]:
from abc import ABC, abstractmethod

# CORRECT: Both depend on abstraction
class Database(ABC):
    @abstractmethod
    def save(self, data: dict): ...

class MySQLDatabase(Database):
    def save(self, data: dict):
        print(f"[MySQL] Saving: {data}")

class MongoDBDatabase(Database):
    def save(self, data: dict):
        print(f"[MongoDB] Saving: {data}")

class UserController:
    def __init__(self, db: Database):  # Depends on abstraction!
        self.db = db

    def create_user(self, name: str):
        self.db.save({"name": name})

# Easily swap implementations
controller1 = UserController(MySQLDatabase())
controller1.create_user("Alice")

controller2 = UserController(MongoDBDatabase())
controller2.create_user("Bob")

---
## Problem 1: Refactor a God Class (SRP)

**Difficulty: Medium**

Refactor this `OrderProcessor` god class following SRP.

In [ ]:
# BEFORE: God class with too many responsibilities
class OrderProcessor:
    def __init__(self, order_data: dict):
        self.order = order_data

    def validate(self) -> bool:
        return bool(self.order.get('items'))

    def calculate_total(self) -> float:
        return sum(item['price'] for item in self.order.get('items', []))

    def apply_discount(self, code: str) -> float:
        total = self.calculate_total()
        if code == "SAVE10":
            return total * 0.9
        return total

    def save_to_db(self):
        print(f"Saving order {self.order.get('id')} to database")

    def send_confirmation_email(self):
        print(f"Sending email for order {self.order.get('id')}")

    def generate_invoice_pdf(self):
        print(f"Generating PDF for order {self.order.get('id')}")

In [ ]:
# AFTER: Separated responsibilities (SRP-compliant)
class Order:
    """Core order data."""
    def __init__(self, order_data: dict):
        self.id = order_data.get('id')
        self.items = order_data.get('items', [])

class OrderValidator:
    """Validates orders."""
    @staticmethod
    def validate(order: Order) -> bool:
        return len(order.items) > 0

class PricingCalculator:
    """Calculates prices and discounts."""
    @staticmethod
    def calculate_total(order: Order) -> float:
        return sum(item['price'] * item.get('qty', 1) for item in order.items)

    @staticmethod
    def apply_discount(total: float, code: str) -> float:
        discounts = {"SAVE10": 0.10, "WELCOME5": 0.05}
        return total * (1 - discounts.get(code, 0))

class OrderRepository:
    """Persists orders."""
    @staticmethod
    def save(order: Order):
        print(f"Saving order {order.id} to database")

class NotificationService:
    """Sends notifications."""
    @staticmethod
    def send_confirmation(order: Order):
        print(f"Sending confirmation for order {order.id}")

class InvoiceService:
    """Generates invoices."""
    @staticmethod
    def generate_pdf(order: Order):
        print(f"Generating invoice PDF for order {order.id}")

# Usage
order = Order({"id": 123, "items": [{"name": "Widget", "price": 25, "qty": 2}]})
if OrderValidator.validate(order):
    total = PricingCalculator.calculate_total(order)
    final = PricingCalculator.apply_discount(total, "SAVE10")
    OrderRepository.save(order)
    NotificationService.send_confirmation(order)
    InvoiceService.generate_pdf(order)
    print(f"Total: ${total}, After discount: ${final:.2f}")

## Problem 2: Plugin System Following OCP

**Difficulty: Medium**

Design a text processing plugin system where new plugins can be added without modifying the processor.

In [ ]:
from abc import ABC, abstractmethod

class TextPlugin(ABC):
    """Abstract plugin interface — open for extension."""
    @abstractmethod
    def process(self, text: str) -> str: ...

    @property
    @abstractmethod
    def name(self) -> str: ...

class UppercasePlugin(TextPlugin):
    name = "uppercase"
    def process(self, text: str) -> str:
        return text.upper()

class ReversePlugin(TextPlugin):
    name = "reverse"
    def process(self, text: str) -> str:
        return text[::-1]

class RemoveSpacesPlugin(TextPlugin):
    name = "no_spaces"
    def process(self, text: str) -> str:
        return ''.join(text.split())

# New plugin — no changes to existing code!
class TitleCasePlugin(TextPlugin):
    name = "title"
    def process(self, text: str) -> str:
        return text.title()

class TextProcessor:
    """Processes text through registered plugins. Closed for modification."""
    def __init__(self):
        self._plugins: dict[str, TextPlugin] = {}

    def register(self, plugin: TextPlugin):
        self._plugins[plugin.name] = plugin

    def process(self, text: str, plugin_chain: list[str]) -> str:
        result = text
        for name in plugin_chain:
            if name in self._plugins:
                result = self._plugins[name].process(result)
        return result

processor = TextProcessor()
processor.register(UppercasePlugin())
processor.register(ReversePlugin())
processor.register(RemoveSpacesPlugin())
processor.register(TitleCasePlugin())  # Added later — no existing code changed!

result = processor.process("hello world", ["title", "reverse"])
print(result)  # Dlrow Olleh

## Key Takeaways

- **SRP**: One reason to change per class — leads to small, focused, testable units.
- **OCP**: Write code that accepts new behavior through extension, not modification.
- **LSP**: Subclasses must fulfill the contract of their base class — no surprises.
- **ISP**: Keep interfaces small; don't force clients to depend on methods they don't use.
- **DIP**: Depend on abstractions (ABCs, Protocols), not concrete classes.
- Following SOLID leads to code that is **testable**, **maintainable**, and **adaptable**.